In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    r"C:\Users\patry\OneDrive\Dokumenty\tcga-brca-survival-analysis\data\clinical.tsv",
    sep="\t",
    low_memory=False
)

df.head()

,project.project_id,cases.case_id,cases.consent_type,cases.days_to_consent,cases.days_to_lost_to_followup,cases.disease_type,cases.index_date,cases.lost_to_followup,cases.primary_site,cases.submitter_id,...,treatments.treatment_duration,treatments.treatment_effect,treatments.treatment_effect_indicator,treatments.treatment_frequency,treatments.treatment_id,treatments.treatment_intent_type,treatments.treatment_or_therapy,treatments.treatment_outcome,treatments.treatment_outcome_duration,treatments.treatment_type
0,TCGA-BRCA,001cef41-ff86-4d3f-a140-a647ac4b10a1,Informed Consent,-34,'--,Ductal and Lobular Neoplasms,Diagnosis,'--,Breast,TCGA-E2-A1IU,...,'--,'--,'--,'--,1b884f21-eb24-467f-aba2-208af17070b9,Adjuvant,no,'--,'--,"Radiation Therapy, NOS"
1,TCGA-BRCA,001cef41-ff86-4d3f-a140-a647ac4b10a1,Informed Consent,-34,'--,Ductal and Lobular Neoplasms,Diagnosis,'--,Breast,TCGA-E2-A1IU,...,'--,'--,'--,'--,27868bc3-23c8-5e85-a0e2-314e6cdf9b2a,Adjuvant,yes,Treatment Ongoing,'--,Hormone Therapy
2,TCGA-BRCA,001cef41-ff86-4d3f-a140-a647ac4b10a1,Informed Consent,-34,'--,Ductal and Lobular Neoplasms,Diagnosis,'--,Breast,TCGA-E2-A1IU,...,'--,'--,'--,'--,aedf144c-6b7b-4d76-a3cb-4271aef10f1d,First-Line Therapy,yes,'--,'--,"Surgery, NOS"
3,TCGA-BRCA,0045349c-69d9-4306-a403-c9c1fa836644,Informed Consent,76,'--,Adenomas and Adenocarcinomas,Diagnosis,'--,Breast,TCGA-A1-A0SB,...,'--,'--,'--,'--,0a534cae-de91-5e77-a3e7-b52d46bd3966,First-Line Therapy,yes,'--,'--,"Surgery, NOS"
4,TCGA-BRCA,00807dae-9f4a-4fd1-aac2-82eb11bf2afb,Informed Consent,19,'--,Adnexal and Skin Appendage Neoplasms,Diagnosis,No,Breast,TCGA-A2-A04W,...,'--,'--,'--,'--,024faa94-ec57-4d14-b919-62dcab409958,Adjuvant,yes,Treatment Ongoing,'--,Bisphosphonate Therapy


In [7]:
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)
# Convert all variants of '--' (even with apostrophes and spaces) to NaN
df = df.replace(r"^\s*'?--'?\s*$", np.nan, regex=True)

# Checking
print(df.shape)
df.head()

(5546, 210)


,project.project_id,cases.case_id,cases.consent_type,cases.days_to_consent,cases.days_to_lost_to_followup,cases.disease_type,cases.index_date,cases.lost_to_followup,cases.primary_site,cases.submitter_id,...,treatments.treatment_duration,treatments.treatment_effect,treatments.treatment_effect_indicator,treatments.treatment_frequency,treatments.treatment_id,treatments.treatment_intent_type,treatments.treatment_or_therapy,treatments.treatment_outcome,treatments.treatment_outcome_duration,treatments.treatment_type
0,TCGA-BRCA,001cef41-ff86-4d3f-a140-a647ac4b10a1,Informed Consent,-34,NaN,Ductal and Lobular Neoplasms,Diagnosis,NaN,Breast,TCGA-E2-A1IU,...,NaN,NaN,NaN,NaN,1b884f21-eb24-467f-aba2-208af17070b9,Adjuvant,no,NaN,NaN,"Radiation Therapy, NOS"
1,TCGA-BRCA,001cef41-ff86-4d3f-a140-a647ac4b10a1,Informed Consent,-34,NaN,Ductal and Lobular Neoplasms,Diagnosis,NaN,Breast,TCGA-E2-A1IU,...,NaN,NaN,NaN,NaN,27868bc3-23c8-5e85-a0e2-314e6cdf9b2a,Adjuvant,yes,Treatment Ongoing,NaN,Hormone Therapy
2,TCGA-BRCA,001cef41-ff86-4d3f-a140-a647ac4b10a1,Informed Consent,-34,NaN,Ductal and Lobular Neoplasms,Diagnosis,NaN,Breast,TCGA-E2-A1IU,...,NaN,NaN,NaN,NaN,aedf144c-6b7b-4d76-a3cb-4271aef10f1d,First-Line Therapy,yes,NaN,NaN,"Surgery, NOS"
3,TCGA-BRCA,0045349c-69d9-4306-a403-c9c1fa836644,Informed Consent,76,NaN,Adenomas and Adenocarcinomas,Diagnosis,NaN,Breast,TCGA-A1-A0SB,...,NaN,NaN,NaN,NaN,0a534cae-de91-5e77-a3e7-b52d46bd3966,First-Line Therapy,yes,NaN,NaN,"Surgery, NOS"
4,TCGA-BRCA,00807dae-9f4a-4fd1-aac2-82eb11bf2afb,Informed Consent,19,NaN,Adnexal and Skin Appendage Neoplasms,Diagnosis,No,Breast,TCGA-A2-A04W,...,NaN,NaN,NaN,NaN,024faa94-ec57-4d14-b919-62dcab409958,Adjuvant,yes,Treatment Ongoing,NaN,Bisphosphonate Therapy


In [10]:
# --- Survival time ---
time_death = pd.to_numeric(df.get('demographic.days_to_death'), errors='coerce')
time_followup = pd.to_numeric(df.get('diagnoses.days_to_last_follow_up'), errors='coerce')
time = time_death.fillna(time_followup)

# --- Event status ---
if 'demographic.vital_status' in df.columns:
    event = df['demographic.vital_status'].str.lower().map(lambda x: 1 if x=='dead' else 0)
else:
    event = pd.Series(np.nan, index=df.index)

# --- Predictors ---
age = pd.to_numeric(df.get('demographic.age_at_index'), errors='coerce')
stage = df.get('diagnoses.ajcc_pathologic_stage', np.nan)
treatment = df.get('treatments.treatment_type', np.nan)

# --- Remove old duplicates if exist ---
for col in ['time', 'event', 'age', 'stage', 'treatment']:
    if col in df.columns:
        df = df.drop(columns=[col])

# --- Combine all into a new DataFrame ---
df_clean = pd.concat([df, time.rename('time'), event.rename('event'),
                      age.rename('age'), stage.rename('stage'), treatment.rename('treatment')], axis=1)

# --- Check ---
print(df_clean[['time','event','age','stage','treatment']].head())

     time  event   age      stage               treatment
0   337.0      0  60.0   Stage IA  Radiation Therapy, NOS
1   337.0      0  60.0   Stage IA         Hormone Therapy
2   337.0      0  60.0   Stage IA            Surgery, NOS
3   259.0      0  70.0    Stage I            Surgery, NOS
4  3102.0      0  50.0  Stage IIB  Bisphosphonate Therapy


In [12]:
# Pokaż wszystkie kolumny w df
print(df.columns.tolist())

['project.project_id', 'cases.case_id', 'cases.consent_type', 'cases.days_to_consent', 'cases.days_to_lost_to_followup', 'cases.disease_type', 'cases.index_date', 'cases.lost_to_followup', 'cases.primary_site', 'cases.submitter_id', 'demographic.age_at_index', 'demographic.age_is_obfuscated', 'demographic.cause_of_death', 'demographic.cause_of_death_source', 'demographic.country_of_birth', 'demographic.country_of_residence_at_enrollment', 'demographic.days_to_birth', 'demographic.days_to_death', 'demographic.demographic_id', 'demographic.education_level', 'demographic.ethnicity', 'demographic.gender', 'demographic.marital_status', 'demographic.occupation_duration_years', 'demographic.population_group', 'demographic.premature_at_birth', 'demographic.race', 'demographic.submitter_id', 'demographic.vital_status', 'demographic.weeks_gestation_at_birth', 'demographic.year_of_birth', 'demographic.year_of_death', 'diagnoses.adrenal_hormone', 'diagnoses.age_at_diagnosis', 'diagnoses.ajcc_clini

In [13]:
# --- Survival variables for analysis ---
# Use the clean DataFrame
df_survival = df_clean[['time', 'event', 'age', 'stage', 'treatment']].dropna()

# Ensure time is numeric
df_survival['time'] = pd.to_numeric(df_survival['time'], errors='coerce')

# Ensure event is numeric (1 = death, 0 = alive/censored)
df_survival['event'] = pd.to_numeric(df_survival['event'], errors='coerce')

# --- Check ---
print("Shape of survival DataFrame:", df_survival.shape)
print(df_survival.head())

# --- Count events ---
event_counts = df_survival['event'].value_counts()
print("\nEvent counts (1 = death, 0 = censored/alive):")
print(event_counts)

Shape of survival DataFrame: (4915, 5)
     time  event   age      stage               treatment
0   337.0      0  60.0   Stage IA  Radiation Therapy, NOS
1   337.0      0  60.0   Stage IA         Hormone Therapy
2   337.0      0  60.0   Stage IA            Surgery, NOS
3   259.0      0  70.0    Stage I            Surgery, NOS
4  3102.0      0  50.0  Stage IIB  Bisphosphonate Therapy

Event counts (1 = death, 0 = censored/alive):
event
0    4331
1     584
Name: count, dtype: int64
